<a href="https://colab.research.google.com/github/anshuman-dataverse/transformer-recipes-imdb/blob/main/transformer_recipes_imdb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformers for Recipe Generation and Sentiment Classification

Part I changes the GPT example so that it generates recipes from the Epicurious dataset, and compares the
results with the LSTM text generator. Part II uses the same GPT block as a sentiment classifier for IMDB
reviews and compares it with an LSTM classifier. All models are trained from scratch.

The assignment gives N_UNITS = 128. For the transformer I used it as the attention key size, and the
feed-forward layer has 256 units, as in the GPT example.

In [ ]:
%matplotlib inline
import os
import re
import json
import time
import string
import textwrap
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, losses, callbacks
from tensorflow.keras.datasets import imdb

VOCAB_SIZE = 10000
MAX_LEN = 200
EMBEDDING_DIM = 100
N_UNITS = 128
VALIDATION_SPLIT = 0.2
SEED = 42
LOAD_MODEL = False
BATCH_SIZE = 32
EPOCHS = 20

N_HEADS = 2
KEY_DIM = N_UNITS
FEED_FORWARD_DIM = 2 * N_UNITS

os.makedirs('models', exist_ok=True)
keras.utils.set_random_seed(SEED)
print('tensorflow', tf.__version__, '| keras', keras.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

## Shared building blocks

These layers come from the GPT example: a token and position embedding, and a transformer block with causal
multi-head attention, a feed-forward layer, residual connections and layer normalisation. Both parts use them.

In [ ]:
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, max_len, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_emb = layers.Embedding(vocab_size, embed_dim)
        self.pos_emb = layers.Embedding(max_len, embed_dim)

    def call(self, x):
        positions = keras.ops.arange(keras.ops.shape(x)[-1])
        return self.token_emb(x) + self.pos_emb(positions)


class TransformerBlock(layers.Layer):
    def __init__(self, num_heads, key_dim, embed_dim, ff_dim, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.attn = layers.MultiHeadAttention(num_heads, key_dim, output_shape=embed_dim)
        self.dropout_1 = layers.Dropout(dropout_rate)
        self.ln_1 = layers.LayerNormalization(epsilon=1e-6)
        self.ffn_1 = layers.Dense(ff_dim, activation='relu')
        self.ffn_2 = layers.Dense(embed_dim)
        self.dropout_2 = layers.Dropout(dropout_rate)
        self.ln_2 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, inputs, training=False):
        attn = self.attn(inputs, inputs, use_causal_mask=True, training=training)
        out_1 = self.ln_1(inputs + self.dropout_1(attn, training=training))
        ffn = self.ffn_2(self.ffn_1(out_1))
        return self.ln_2(out_1 + self.dropout_2(ffn, training=training))


def plot_history(histories, title):
    fig, ax = plt.subplots(figsize=(7, 3.6))
    for name, h in histories.items():
        line, = ax.plot(range(1, len(h['loss']) + 1), h['loss'], label=f'{name} train')
        ax.plot(range(1, len(h['val_loss']) + 1), h['val_loss'], '--', color=line.get_color(), label=f'{name} val')
    ax.set_xlabel('epoch')
    ax.set_ylabel('loss')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


class BestWeights(callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.best, self.best_epoch, self.weights = np.inf, None, None

    def on_epoch_end(self, epoch, logs=None):
        if logs['val_loss'] < self.best:
            self.best, self.best_epoch = logs['val_loss'], epoch + 1
            self.weights = self.model.get_weights()


class EpochTimer(callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.times = []

    def on_epoch_begin(self, epoch, logs=None):
        self._t = time.time()

    def on_epoch_end(self, epoch, logs=None):
        self.times.append(time.time() - self._t)


def fit_or_load(model, path, train_ds, val_ds, extra_callbacks=()):
    timer = EpochTimer()
    if LOAD_MODEL and os.path.exists(path):
        model.load_weights(path)
        return None, timer
    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                     callbacks=[timer, *extra_callbacks], verbose=2)
    model.save_weights(path)
    return hist.history, timer

# Part I: Recipe generation

## 1. Data

Each recipe is turned into one string in the same format as the LSTM example: `Recipe for <title> | <directions>`.
Punctuation is split off into separate tokens. The recipes are shuffled with the seed and split 80/20 into
training and validation sets.

In [ ]:
def find_recipes_file():
    import glob, subprocess, sys
    hits = glob.glob('data/**/full_format_recipes.json', recursive=True)
    if hits:
        return hits[0]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'kagglehub'], check=True)
    import kagglehub
    dataset_dir = kagglehub.dataset_download('hugodarwood/epirecipes')
    return glob.glob(os.path.join(dataset_dir, '**', 'full_format_recipes.json'), recursive=True)[0]


DATA_PATH = find_recipes_file()
print('recipes file:', DATA_PATH)
with open(DATA_PATH) as f:
    recipe_data = json.load(f)

filtered = [r for r in recipe_data if r.get('title') and r.get('directions')]
recipes = ['Recipe for ' + r['title'] + ' | ' + ' '.join(r['directions']) for r in filtered]


def pad_punctuation(s):
    s = re.sub(f'([{re.escape(string.punctuation)}])', r' \1 ', s)
    return re.sub(' +', ' ', s).strip()


recipes = [pad_punctuation(r) for r in recipes]
rng = np.random.default_rng(SEED)
order = rng.permutation(len(recipes))
n_val = int(len(recipes) * VALIDATION_SPLIT)
val_text = [recipes[i] for i in order[:n_val]]
train_text = [recipes[i] for i in order[n_val:]]

lengths = np.array([len(r.split()) for r in recipes])
print(f'{len(recipe_data)} raw recipes -> {len(recipes)} with a title and directions')
print(f'train {len(train_text)} | validation {len(val_text)}')
print(f'tokens per recipe: median {np.median(lengths):.0f}, '
      f'{(lengths > MAX_LEN).mean():.1%} longer than MAX_LEN={MAX_LEN}')
print()
print(textwrap.fill(train_text[0], 110))

In [ ]:
vectorize_layer = layers.TextVectorization(standardize='lower', max_tokens=VOCAB_SIZE,
                                           output_mode='int', output_sequence_length=MAX_LEN + 1)
vectorize_layer.adapt(tf.data.Dataset.from_tensor_slices(train_text).batch(256))
vocab = vectorize_layer.get_vocabulary()
word_to_index = {w: i for i, w in enumerate(vocab)}


def to_ids(texts):
    return np.concatenate([vectorize_layer(tf.constant(texts[i:i + 1024])).numpy()
                           for i in range(0, len(texts), 1024)])


train_ids, val_ids = to_ids(train_text), to_ids(val_text)


def lm_dataset(ids, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((ids[:, :-1], ids[:, 1:]))
    if shuffle:
        ds = ds.shuffle(len(ids), seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


lm_train_ds, lm_val_ds = lm_dataset(train_ids, True), lm_dataset(val_ids, False)
print('vocabulary', len(vocab), '| first tokens:', vocab[:12])
print('unknown-token rate in validation:',
      f'{(val_ids == 1).sum() / (val_ids > 0).sum():.2%}')

## 2. Models

The GPT model is the example model with the recipe vocabulary: an embedding, one transformer block and a
softmax over the 10,000 words at every position. The LSTM model is the one from the autoregressive LSTM
example: an embedding, an LSTM with 128 units and the same softmax output. Both use the same data, loss and
optimiser.

In [ ]:
def build_gpt():
    inputs = layers.Input(shape=(None,), dtype='int32')
    x = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBEDDING_DIM)(inputs)
    x = TransformerBlock(N_HEADS, KEY_DIM, EMBEDDING_DIM, FEED_FORWARD_DIM)(x)
    outputs = layers.Dense(VOCAB_SIZE, activation='softmax')(x)
    model = keras.Model(inputs, outputs, name='gpt_recipes')
    model.compile('adam', loss=losses.SparseCategoricalCrossentropy())
    return model


def build_lstm_lm():
    inputs = layers.Input(shape=(None,), dtype='int32')
    x = layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM)(inputs)
    x = layers.LSTM(N_UNITS, return_sequences=True)(x)
    outputs = layers.Dense(VOCAB_SIZE, activation='softmax')(x)
    model = keras.Model(inputs, outputs, name='lstm_recipes')
    model.compile('adam', loss=losses.SparseCategoricalCrossentropy())
    return model


keras.utils.set_random_seed(SEED)
gpt = build_gpt()
lstm_lm = build_lstm_lm()
gpt.summary()
lstm_lm.summary()

## 3. Training

Both models are trained for 20 epochs with a batch size of 32. The Keras loss also counts the padding after the
end of each recipe, which is easy to predict, so after training I also compute the loss on real tokens only and
report it as perplexity.

In [ ]:
keras.utils.set_random_seed(SEED)
gpt_hist, gpt_timer = fit_or_load(gpt, 'models/gpt_recipes.weights.h5', lm_train_ds, lm_val_ds)

In [ ]:
keras.utils.set_random_seed(SEED)
lstm_lm_hist, lstm_lm_timer = fit_or_load(lstm_lm, 'models/lstm_recipes.weights.h5', lm_train_ds, lm_val_ds)

In [ ]:
if gpt_hist and lstm_lm_hist:
    plot_history({'GPT': gpt_hist, 'LSTM': lstm_lm_hist}, 'Recipe language models')


def masked_perplexity(model, ds):
    total, count, correct = 0.0, 0, 0
    for x, y in ds:
        p = model(x, training=False)
        mask = keras.ops.cast(y > 0, 'float32')
        ce = losses.sparse_categorical_crossentropy(y, p)
        total += float(keras.ops.sum(ce * mask))
        count += float(keras.ops.sum(mask))
        correct += float(keras.ops.sum(keras.ops.cast(keras.ops.equal(keras.ops.cast(keras.ops.argmax(p, -1), 'int32'), keras.ops.cast(y, 'int32')), 'float32') * mask))
    return np.exp(total / count), correct / count


lm_rows = []
for name, model, hist, timer in [('GPT', gpt, gpt_hist, gpt_timer), ('LSTM', lstm_lm, lstm_lm_hist, lstm_lm_timer)]:
    ppl, acc = masked_perplexity(model, lm_val_ds)
    lm_rows.append({'model': name, 'params': model.count_params(),
                    'final train loss': hist['loss'][-1], 'final val loss': hist['val_loss'][-1],
                    'best val epoch': int(np.argmin(hist['val_loss'])) + 1,
                    'val perplexity (real tokens)': ppl, 'val next-token acc': acc,
                    'sec / epoch': np.mean(timer.times)})
lm_df = pd.DataFrame(lm_rows).set_index('model')
lm_df.round(3)

### Observations

The GPT had a lower validation loss than the LSTM in every epoch (1.641 against 1.744 at epoch 20). On real
tokens only, its validation perplexity is 9.3 and the LSTM's is 10.7. Neither curve has flattened by epoch 20,
so both models would probably still improve with more epochs. The GPT also trained faster, 19.6 s per epoch
against 28.1 s on the Colab GPU, because attention processes all positions at once while the LSTM has to go
through them one by one.

## 4. Text generation

Generation works the same way as in the example. The model predicts the next token from the prompt, the
probabilities are adjusted with the temperature T, a token is sampled and added to the text, and this repeats
until the model outputs padding (end of recipe) or the text reaches 200 tokens.

$$p_i^{(T)} = \frac{p_i^{1/T}}{\sum_j p_j^{1/T}}$$

A low temperature makes the most likely tokens even more likely, and a high temperature flattens the
distribution.

In [ ]:
gen_rng = np.random.default_rng(SEED)


def make_steps(model):
    body = keras.Model(model.input, model.layers[-2].output)
    head = model.layers[-1]

    @tf.function(reduce_retracing=True)
    def full(x):
        return model(x, training=False)

    @tf.function(reduce_retracing=True)
    def next_token(x, t):
        return head(body(x, training=False)[:, t - 1, :])
    return full, next_token


steps, next_steps = {}, {}
for _name, _model in [('GPT', gpt), ('LSTM', lstm_lm)]:
    steps[_name], next_steps[_name] = make_steps(_model)


def generate(model_name, prompt, n, temperature, max_tokens=MAX_LEN):
    prompt_ids = [word_to_index.get(w, 1) for w in pad_punctuation(prompt).lower().split()]
    seqs = np.zeros((n, max_tokens), dtype='int32')
    seqs[:, :len(prompt_ids)] = prompt_ids
    ended = np.zeros(n, dtype=bool)
    for t in range(len(prompt_ids), max_tokens):
        probs = next_steps[model_name](tf.constant(seqs), tf.constant(t)).numpy().astype('float64')
        logits = np.log(probs + 1e-12) / temperature
        nxt = np.argmax(logits + gen_rng.gumbel(size=logits.shape), axis=-1)
        nxt[ended] = 0
        seqs[:, t] = nxt
        ended |= nxt == 0
        if ended.all():
            break
    out = []
    for s in seqs:
        s = list(s)
        out.append(s[:s.index(0)] if 0 in s else s)
    return out


def detok(ids):
    text = ' '.join(vocab[i] if i < len(vocab) else '[UNK]' for i in ids)
    return re.sub(r'\s+([.,;:!?)%])', r'\1', text).replace('( ', '(')


def show(model_name, prompt, temperature, n=1, width=110):
    for ids in generate(model_name, prompt, n, temperature):
        print(textwrap.fill(detok(ids), width))
        print()

In [ ]:
TEMPERATURES = [0.2, 0.5, 1.0, 1.5]
PROMPTS = ['recipe for roasted vegetables |', 'recipe for chocolate', 'recipe for']

for prompt in PROMPTS:
    for model_name in ['GPT', 'LSTM']:
        print('=' * 110)
        print(f'{model_name}  |  prompt: "{prompt}"')
        print('=' * 110)
        for T in TEMPERATURES:
            print(f'--- T = {T}')
            show(model_name, prompt, T)

### Next-token probabilities at different temperatures

The GPT's prediction for the word after `recipe for roasted vegetables | preheat oven to`, at each temperature.

In [ ]:
probe = 'recipe for roasted vegetables | preheat oven to'
probe_ids = [word_to_index.get(w, 1) for w in probe.split()]
p = gpt(np.array([probe_ids]), training=False).numpy()[0, -1].astype('float64')

fig, axes = plt.subplots(1, len(TEMPERATURES), figsize=(14, 3), sharey=True)
for ax, T in zip(axes, TEMPERATURES):
    q = p ** (1 / T)
    q /= q.sum()
    top = np.argsort(q)[::-1][:8]
    ax.barh([vocab[i] if i < len(vocab) else '[UNK]' for i in top][::-1], q[top][::-1], color='tab:blue')
    ax.set_title(f'T = {T}  (top-1 = {q[top[0]]:.2f})', fontsize=10)
    ax.set_xlim(0, 1)
plt.suptitle(f'GPT next-token probabilities after "{probe}"', fontsize=11)
plt.tight_layout()
plt.show()

### Effect of temperature

In the chart, the GPT gives `450°f` a probability of 0.36 at T = 1. At T = 0.2 this goes up to 0.98, so the
model almost always picks the same word, and at T = 1.5 it goes down to 0.22.

The samples show the same pattern. At T = 0.2 the sentences are correct but the text repeats itself. The GPT
writes "carrots, celery, celery, and celery" and "grilled shrimp with shrimp and shrimp", and its chocolate sauce
uses the word chocolate 19 times. The LSTM repeats whole phrases instead: "beating well after each addition"
appears five times in a row in its chocolate cake.

At T = 0.5 the recipes are the most readable. The GPT's grilled pork chops recipe makes sense from start to
finish: marinate the chops, cook them to 145°F, let the meat rest, then make the onion sauce. At T = 1.0 each
sentence still sounds like a recipe, but ingredients appear at random; the roasted vegetables recipe ends up
with beef and drumsticks. At T = 1.5 the text stops making sense ("grate enough chives into an litchis salsa",
"devein plums").

### Measuring the samples

For each model and temperature I generate 50 recipes from the prompt `recipe for`, so the model also has to
come up with the title. The measures are:

| measure | meaning |
|---|---|
| length | average number of tokens before the recipe ends |
| ended | share of recipes the model ended itself instead of reaching 200 tokens |
| repeated 4-grams | share of 4-grams in a recipe that already appeared earlier in the same recipe |
| distinct-1 / distinct-2 | unique words / word pairs divided by the total, over all 50 recipes |
| novel 4-grams | share of generated 4-grams that never appear in the training recipes |
| novel titles | share of generated titles that are not in the training set |
| self-PPL / cross-PPL | perplexity of the samples under the model that wrote them / under the other model |

In [ ]:
N_SAMPLES = 50
EVAL_TEMPERATURES = [0.2, 0.5, 0.75, 1.0, 1.25, 1.5]


def ngrams(seq, n):
    return [tuple(seq[i:i + n]) for i in range(len(seq) - n + 1)]


train_4grams = set()
for row in train_ids:
    seq = row[row > 0].tolist()
    train_4grams.update(ngrams(seq, 4))
bar = word_to_index['|']
train_titles = {tuple(r[:r.index(bar)]) for r in (list(x[x > 0]) for x in train_ids) if bar in r}


def text_perplexity(model_name, samples):
    total, count = 0.0, 0
    for s in samples:
        s = s[:MAX_LEN]
        x = np.zeros((1, MAX_LEN), dtype='int32')
        x[0, :len(s)] = s
        p = steps[model_name](tf.constant(x)).numpy()[0]
        idx = np.arange(2, len(s))
        total -= np.log(p[idx - 1, np.array(s)[idx]] + 1e-12).sum()
        count += len(idx)
    return np.exp(total / count)


def sample_metrics(model_name, samples):
    all_1 = [g for s in samples for g in ngrams(s, 1)]
    all_2 = [g for s in samples for g in ngrams(s, 2)]
    all_4 = [g for s in samples for g in ngrams(s, 4)]
    rep = [1 - len(set(ngrams(s, 4))) / max(len(ngrams(s, 4)), 1) for s in samples]
    titles = [tuple(s[:s.index(bar)]) for s in samples if bar in s]
    other = 'LSTM' if model_name == 'GPT' else 'GPT'
    return {'length': np.mean([len(s) for s in samples]),
            'ended': np.mean([len(s) < MAX_LEN for s in samples]),
            'repeated 4-grams': np.mean(rep),
            'distinct-1': len(set(all_1)) / len(all_1),
            'distinct-2': len(set(all_2)) / len(all_2),
            'novel 4-grams': np.mean([g not in train_4grams for g in all_4]),
            'novel titles': np.mean([t not in train_titles for t in titles]) if titles else np.nan,
            'self-PPL': text_perplexity(model_name, samples),
            'cross-PPL': text_perplexity(other, samples)}


rows, sample_bank = [], {}
for model_name in ['GPT', 'LSTM']:
    for T in EVAL_TEMPERATURES:
        samples = generate(model_name, 'recipe for', N_SAMPLES, T)
        sample_bank[(model_name, T)] = samples
        rows.append({'model': model_name, 'T': T, **sample_metrics(model_name, samples)})

human = [list(r[r > 0]) for r in val_ids[:N_SAMPLES]]
ref = sample_metrics('GPT', human)
ref['self-PPL'], ref['cross-PPL'] = text_perplexity('GPT', human), text_perplexity('LSTM', human)
rows.append({'model': 'human (val)', 'T': np.nan, **ref})
gen_df = pd.DataFrame(rows)
gen_df.rename(columns={'self-PPL': 'self-PPL (GPT for human)', 'cross-PPL': 'cross-PPL (LSTM for human)'}).round(3)

In [ ]:
metrics = [('repeated 4-grams', 'repetition within a sample'), ('distinct-2', 'diversity across samples'),
           ('novel 4-grams', 'novelty vs training data'), ('cross-PPL', 'perplexity under the other model (log)')]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.4))
for ax, (m, title) in zip(axes, metrics):
    for name, style in [('GPT', 'o-'), ('LSTM', 's-')]:
        d = gen_df[gen_df.model == name]
        ax.plot(d['T'], d[m], style, label=name)
    ax.axhline(gen_df.loc[gen_df.model == 'human (val)', m].item(), color='gray', ls=':', label='human')
    ax.set_title(f'{m}\n{title}', fontsize=10)
    ax.set_xlabel('temperature')
    ax.grid(alpha=0.3)
    if m == 'cross-PPL':
        ax.set_yscale('log')
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. Comparison with the LSTM

The dotted line in each plot is the value for 50 real recipes from the validation set. For the real recipes,
the table gives the perplexity under the GPT (10.3) and under the LSTM (11.9).

**Coherence.** The GPT recipes are more coherent, especially at higher temperatures. At T = 1.0 the LSTM gives
the GPT's samples a perplexity of 18.2, about 1.5 times its value for real recipes. The GPT gives the LSTM's
samples 43.4, about 4.2 times its value for real recipes. The two models are not equally good judges, so this
is only a rough comparison, but the GPT comes out ahead at every temperature from 1.0 up. Reading the samples
gives the same impression. With the roasted vegetables prompt at T = 0.5 the GPT keeps talking about roasting
vegetables, while the LSTM moves on to lamb, pasta and shrimp after a few sentences, and its "chocolate-glazed
salmon" never uses any salmon. I think this is because of attention: the GPT can look back at the title at
every step, while the LSTM has to keep it in its hidden state. The downside is that the GPT repeats words more
at low temperature (16.5% repeated 4-grams against 13.7% at T = 0.2).

**Creativity.** Both models make up new dishes: 90 to 100% of the generated titles are not in the training
set. Some are believable, like "sauteed asparagus with roasted red bell pepper", and some are strange, like
"chocolate-mashed sweet potatoes". In the recipe text, the share of 4-grams that never appear in the training
data reaches the level of real recipes (31%) at about T = 0.75 for both models. Above that the LSTM produces
more new 4-grams (58% against 47% at T = 1.0), but its high cross-perplexity shows that most of these are
nonsense rather than new ideas.

**Diversity.** The variety between samples (distinct-2) goes up with temperature in almost the same way for
both models, and both reach the level of real recipes at about T = 1.0. The LSTM is slightly more varied at
high temperatures because its predictions are less confident to begin with.

Overall the GPT writes better recipes at the same temperature, while the LSTM is only slightly more varied.
For both models a temperature of about 0.5 to 0.75 gave the best results.

# Part II: Sentiment classification

## 1. Data

The IMDB reviews are already encoded as integers. With `num_words=10000`, index 0 is padding, 1 is the start
token, 2 is an unknown word, and real words start at 3. Every review is padded or cut to 200 tokens. Padding is
added at the front and long reviews keep their last 200 tokens, which usually contain the reviewer's verdict.
20% of the 25,000 training reviews are used for validation, and the 25,000 test reviews are only used at the end.

In [ ]:
(x_train_full, y_train_full), (x_test_raw, y_test) = imdb.load_data(num_words=VOCAB_SIZE)
raw_test_len = np.array([len(s) for s in x_test_raw])

x_train_full = keras.utils.pad_sequences(x_train_full, maxlen=MAX_LEN)
x_test = keras.utils.pad_sequences(x_test_raw, maxlen=MAX_LEN)

perm = np.random.default_rng(SEED).permutation(len(x_train_full))
n_val = int(len(perm) * VALIDATION_SPLIT)
x_val, y_val = x_train_full[perm[:n_val]], y_train_full[perm[:n_val]]
x_train, y_train = x_train_full[perm[n_val:]], y_train_full[perm[n_val:]]

print(f'train {x_train.shape} | val {x_val.shape} | test {x_test.shape}')
print(f'positive share: train {y_train.mean():.3f}, val {y_val.mean():.3f}, test {y_test.mean():.3f}')
print(f'test reviews longer than {MAX_LEN} tokens: {(raw_test_len > MAX_LEN).mean():.1%} '
      f'(median length {np.median(raw_test_len):.0f})')


def clf_dataset(x, y, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        ds = ds.shuffle(len(x), seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


clf_train_ds = clf_dataset(x_train, y_train, True)
clf_val_ds = clf_dataset(x_val, y_val, False)

## 2. Models

The GPT classifier is the Part I model with a different output. The softmax that predicted the next word is
replaced by an average over the 200 positions and a single sigmoid unit. The embedding and the transformer
block are the same as in Part I.

The LSTM classifier is an embedding, an LSTM with 128 units, and a sigmoid output on the last hidden state.

In [ ]:
def build_gpt_classifier():
    inputs = layers.Input(shape=(MAX_LEN,), dtype='int32')
    x = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBEDDING_DIM)(inputs)
    x = TransformerBlock(N_HEADS, KEY_DIM, EMBEDDING_DIM, FEED_FORWARD_DIM)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.1)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs, name='gpt_classifier')
    model.compile('adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


def build_lstm_classifier():
    inputs = layers.Input(shape=(MAX_LEN,), dtype='int32')
    x = layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM)(inputs)
    x = layers.LSTM(N_UNITS)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs, name='lstm_classifier')
    model.compile('adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


keras.utils.set_random_seed(SEED)
gpt_clf = build_gpt_classifier()
gpt_clf.summary()

## 3. Training

Each model is trained for 20 epochs. I also keep the weights from the epoch with the lowest validation loss,
so the test set can be scored with both the final weights and the best ones.

In [ ]:
clf_models, clf_hist, clf_timers, clf_best = {}, {}, {}, {}
for name, builder in [('GPT', build_gpt_classifier), ('LSTM', build_lstm_classifier)]:
    print(f'\n### {name}')
    keras.utils.set_random_seed(SEED)
    model = builder()
    best = BestWeights()
    path = 'models/' + re.sub(r'\W+', '_', name.lower()).strip('_') + '.weights.h5'
    hist, timer = fit_or_load(model, path, clf_train_ds, clf_val_ds, [best])
    clf_models[name], clf_hist[name], clf_timers[name], clf_best[name] = model, hist, timer, best

In [ ]:
plot_history({k: v for k, v in clf_hist.items() if v}, 'IMDB training and validation loss')

fig, ax = plt.subplots(figsize=(7, 3.4))
for name, h in clf_hist.items():
    line, = ax.plot(range(1, EPOCHS + 1), h['accuracy'], label=f'{name} train')
    ax.plot(range(1, EPOCHS + 1), h['val_accuracy'], '--', color=line.get_color(), label=f'{name} val')
ax.set_xlabel('epoch')
ax.set_ylabel('accuracy')
ax.set_title('IMDB training and validation accuracy')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

loss_table = pd.DataFrame({f'{name} {split}': h[key] for name, h in clf_hist.items()
                           for split, key in [('train', 'loss'), ('val', 'val_loss')]},
                          index=pd.RangeIndex(1, EPOCHS + 1, name='epoch'))
loss_table.round(4)

### Training and validation loss

| model | lowest val loss (epoch) | val loss at epoch 20 | train loss at epoch 20 |
|---|---|---|---|
| GPT | 0.300 (2) | 1.383 | 0.011 |
| LSTM | 0.331 (3) | 0.889 | 0.004 |

Both models overfit early. The training loss keeps going down towards zero, but the validation loss is lowest
at epoch 2 or 3 and then rises, to 4.6 times its minimum for the GPT and 2.7 times for the LSTM. Validation
accuracy only drops by about 2 points over the same epochs, so most of the increase in loss comes from the
models becoming very confident on the reviews they get wrong. The GPT's validation loss also jumps around more
between epochs.

## 4. Test accuracy

In [ ]:
rows, test_probs = [], {}
for name, model in clf_models.items():
    h, best = clf_hist[name], clf_best[name]
    final_loss, final_acc = model.evaluate(x_test, y_test, batch_size=256, verbose=0)
    final_weights = model.get_weights()
    model.set_weights(best.weights)
    best_loss, best_acc = model.evaluate(x_test, y_test, batch_size=256, verbose=0)
    test_probs[name] = model.predict(x_test, batch_size=256, verbose=0).ravel()
    model.set_weights(final_weights)
    rows.append({'model': name, 'params': model.count_params(),
                 'final train loss': h['loss'][-1], 'final val loss': h['val_loss'][-1],
                 'min val loss': min(h['val_loss']), 'best epoch': best.best_epoch,
                 'test acc (epoch 20)': final_acc, 'test acc (best epoch)': best_acc,
                 'test loss (epoch 20)': final_loss, 'sec / epoch': np.mean(clf_timers[name].times)})
clf_df = pd.DataFrame(rows).set_index('model')
clf_df.round(4)

The test set has 25,000 reviews, so each accuracy has an uncertainty of about ±0.2 points, and these results
come from a single seed.

With the best-epoch weights the LSTM gets 86.2% and the GPT 85.7%. With the epoch 20 weights the LSTM gets
84.1% and the GPT 82.2%. Stopping at the best epoch would have helped both models more than the difference
between them.

### Accuracy by review length

Test accuracy with the best-epoch weights, split by the original length of the review.

In [ ]:
bins = [0, 100, 200, 400, np.inf]
labels = ['<100', '100-200', '200-400', '>400']
bucket = pd.cut(raw_test_len, bins=bins, labels=labels)
by_len = pd.DataFrame({name: pd.Series((p > 0.5) == y_test).groupby(bucket, observed=True).mean()
                       for name, p in test_probs.items()})
by_len['n reviews'] = pd.Series(bucket).value_counts().reindex(labels).values
by_len.round(4)

In [ ]:
ax = by_len.drop(columns='n reviews').plot.bar(figsize=(7, 3.4), rot=0, width=0.8)
ax.set_ylim(0.75, 0.95)
ax.set_xlabel('original review length (tokens)')
ax.set_ylabel('test accuracy')
ax.set_title('Test accuracy by review length (best-epoch weights)')
ax.grid(alpha=0.3, axis='y')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

agree = np.mean((test_probs['GPT'] > 0.5) == (test_probs['LSTM'] > 0.5))
print(f'GPT and LSTM give the same label on {agree:.1%} of test reviews')

## 5. Analysis

The transformer did not do better than the LSTM. It was 0.5 points behind with the best-epoch weights and
1.9 points behind after 20 epochs, and the two models gave the same label to 87.9% of the test reviews.

**Model architecture.** The GPT uses a causal mask, so each word can only attend to the words before it. That
is needed for predicting the next word in Part I, but for classifying a review it means the first words are
processed without seeing the rest of the review. The model is also small, with one transformer block and two
heads, and it is trained from scratch on 20,000 reviews. Transformers usually need more layers or pretraining
to do well, and pretraining was not allowed here. The GPT also has to learn word order through its 200 position
embeddings, while the LSTM gets word order for free because it reads the words in sequence. Finally, sentiment
in IMDB reviews depends a lot on single words such as "great", "boring" or "waste", which both models pick up
easily, so there is not much room for one architecture to beat the other.

**Long-range dependencies.** In attention every word is directly connected to every other word, while the LSTM
has to pass information through up to 200 steps. The results by review length show this a little. The LSTM is
better on short reviews (87.6% against 86.6% under 100 tokens), but the GPT is better on reviews longer than 400
tokens (83.0% against 82.3%). From the shortest to the longest group the LSTM loses 5.3 points and the GPT 3.6.
The bigger limit for both models is MAX_LEN = 200: 41% of the test reviews are longer than that and lose
everything before their last 200 tokens, which is why both models do worse on long reviews.

**Training behaviour.** The GPT learned faster at the start. It reached 86.7% validation accuracy after one
epoch, while the LSTM stayed around 84% for the first two epochs. It also overfit more: its validation loss
reached 1.38 by epoch 20 against 0.89 for the LSTM. One epoch took about the same time for both models on the
GPU, around 7 seconds. With only 20,000 training reviews, 20 epochs is too many for both models, and early
stopping or more dropout would help more than changing the architecture.

In conclusion, a small GPT trained from scratch is not better than an LSTM for this task. To get a clear
advantage it would probably need attention without the causal mask, longer inputs, early stopping and
pretraining.